In [ ]:
import re
import torch
import pickle
import nltk
import pandas as pd
import numpy as np
import torch.nn as nn
import torch.optim as optim
from nltk.corpus import stopwords
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from torch.utils.data import Dataset, DataLoader
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [ ]:
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [ ]:
labels=[]
texts=[]

with open("/content/train.ft.txt",'r',encoding='utf-8') as file:
  for line in file:
    parts=line.strip().split(' ',1)
    if len(parts)==2:
      label=parts[0].replace('__label__','')
      text=parts[1]

      labels.append(int(label))
      texts.append(text)
df=pd.DataFrame({
    'label':labels,
    'text':texts
})

In [ ]:
df['sentiment']=df['label'].map({1:'negative',2:'positive'})

## Text Preprocessing & Vocabulary

In [ ]:
df['text'].head()

,text
0,Stuning even for the non-gamer: This sound tra...
1,The best soundtrack ever to anything.: I'm rea...
2,Amazing!: This soundtrack is my favorite music...
3,Excellent Soundtrack: I truly like this soundt...
4,"Remember, Pull Your Jaw Off The Floor After He..."


In [ ]:
def clean_text(text,remove_stopwords=False):
  text=text.lower()
  text=re.sub(r'[^a-z\s]','',text)
  text=re.sub(r'\s+',' ', text).strip()
  if remove_stopwords:
    stop_words=set(stopwords.words('english'))
    words=text.split()
    text=' '.join([word for word in words if word not in stop_words])
  return text

In [ ]:
df['clean_text']=df['text'].apply(lambda x: clean_text(x,remove_stopwords=True))

In [ ]:
vocab={"<PAD>":0,"<UNK>":1}
all_words=[]
for text in df['clean_text']:
  all_words.extend(text.split())
word_counts=Counter(all_words)
for word,_ in word_counts.items():
  vocab[word]=len(vocab)
print(f"Unique vocabulary size:{len(vocab)}")

Unique vocabulary size:345714


## Numerical Sequence and Padding

In [ ]:
def text_to_sequence(text,vocab):
  return [vocab.get(word,vocab['<UNK>']) for word in text.split() ]
df['sequence']=df['clean_text'].apply(lambda z: text_to_sequence(z,vocab))

max_len=12

def pad_sequence(seq,max_len):
  if len(seq)<max_len:
    return seq+[vocab['<PAD>']]*(max_len-len(seq))
  else:
    return seq[:max_len]
df['padding']=df['sequence'].apply(lambda s: pad_sequence(s,max_len))


## Create PyTorch Dataset ও DataLoader

In [ ]:
class SentimentDataset(Dataset):
  def __init__(self,sequence,labels):
    self.sequence=torch.tensor(sequence,dtype=torch.long)
    self.labels=torch.tensor(labels,dtype=torch.long)-1
  def __len__(self):
    return len(self.labels)
  def __getitem__(self, idx):
    return self.sequence[idx],self.labels[idx]


## Train test split

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(list(df['padding']),list(df['label']),test_size=0.2,random_state=42)

train_dataset=SentimentDataset(X_train,y_train)
test_dataset=SentimentDataset(X_test,y_test)

train_loader=DataLoader(train_dataset,batch_size=32,shuffle=True)
test_loader=DataLoader(test_dataset,batch_size=32,shuffle=False)

## Bi-LSTM Model Architechture

In [ ]:
class BiLSTMClassification(nn.Module):
  def __init__(self, vocab_size, embedding_dim, hidden_size, output_dim, pad_idx):
    super().__init__()

    self.embedding=nn.Embedding(vocab_size,embedding_dim,padding_idx=pad_idx)
    self.lstm=nn.LSTM(embedding_dim,hidden_size,batch_first=True,bidirectional=True)
    self.fc=nn.Linear(hidden_size*2,output_dim)
    self.dropout=nn.Dropout(0.5)
  def forward(self,text):
    embedded=self.dropout(self.embedding(text))
    lstm_out,(hidden,cell)=self.lstm(embedded)
    hidden_cat=torch.cat((hidden[-2,:,:],hidden[-1,:,:]),dim=1)
    dense_out=self.dropout(hidden_cat)
    output=self.fc(dense_out)
    return output

## Model Initialization

In [ ]:
EPOCHS=15
VOCAB_SIZE=len(vocab)
EMBEDDING_DIM = 64
HIDDEN_SIZE = 32
OUTPUT_DIM = 2 # Positive / Negative
PAD_IDX = vocab["<PAD>"]
model=BiLSTMClassification(VOCAB_SIZE,EMBEDDING_DIM,HIDDEN_SIZE,OUTPUT_DIM,PAD_IDX)
criterion=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters(),lr=0.001)

## Model Training

In [ ]:
model = model.to(device)

for epoch in range(EPOCHS):
  model.train()
  epoch_loss = 0
  correct_train = 0
  total_train = 0

  for batch_x, batch_y in train_loader:
    batch_x, batch_y = batch_x.to(device), batch_y.to(device)
    optimizer.zero_grad()

    outputs = model(batch_x)
    loss = criterion(outputs, batch_y)
    loss.backward()

    optimizer.step()

    epoch_loss += loss.item()
    _, predicted = torch.max(outputs, 1)
    total_train += batch_y.size(0)
    correct_train += (predicted == batch_y).sum().item()

  train_loss = epoch_loss / len(train_loader)
  train_acc = 100 * correct_train / total_train


  model.eval()
  val_loss = 0
  correct_val = 0
  total_val = 0

  with torch.no_grad():
    for batch_x, batch_y in test_loader:
      batch_x, batch_y = batch_x.to(device), batch_y.to(device)
      outputs = model(batch_x)
      loss = criterion(outputs, batch_y)

      val_loss += loss.item()

      _, predicted = torch.max(outputs, 1)
      total_val += batch_y.size(0)
      correct_val += (predicted == batch_y).sum().item()

  val_loss = val_loss / len(test_loader)
  val_acc = 100 * correct_val / total_val

  print(f"Epoch [{epoch+1}/{EPOCHS}] | "
          f"Train Loss: {train_loss:.4f} - Train Acc: {train_acc:.2f}% | "
          f"Val Loss: {val_loss:.4f} - Val Acc: {val_acc:.2f}%")

Epoch [1/15] | Train Loss: 0.5056 - Train Acc: 74.39% | Val Loss: 0.3947 - Val Acc: 82.36%
Epoch [2/15] | Train Loss: 0.4033 - Train Acc: 81.53% | Val Loss: 0.3717 - Val Acc: 83.94%
Epoch [3/15] | Train Loss: 0.3714 - Train Acc: 83.38% | Val Loss: 0.3585 - Val Acc: 84.60%
Epoch [4/15] | Train Loss: 0.3512 - Train Acc: 84.58% | Val Loss: 0.3638 - Val Acc: 84.92%
Epoch [5/15] | Train Loss: 0.3374 - Train Acc: 85.27% | Val Loss: 0.3539 - Val Acc: 85.18%
Epoch [6/15] | Train Loss: 0.3262 - Train Acc: 85.87% | Val Loss: 0.3587 - Val Acc: 85.33%
Epoch [7/15] | Train Loss: 0.3172 - Train Acc: 86.44% | Val Loss: 0.3672 - Val Acc: 85.36%
Epoch [8/15] | Train Loss: 0.3102 - Train Acc: 86.72% | Val Loss: 0.3604 - Val Acc: 85.45%
Epoch [9/15] | Train Loss: 0.3023 - Train Acc: 87.14% | Val Loss: 0.3599 - Val Acc: 85.50%
Epoch [10/15] | Train Loss: 0.2967 - Train Acc: 87.34% | Val Loss: 0.3591 - Val Acc: 85.44%
Epoch [11/15] | Train Loss: 0.2900 - Train Acc: 87.67% | Val Loss: 0.3592 - Val Acc: 85.4

## Real-time Prediction Check

In [ ]:
def predict_sentiment(text,model,vocab,max_len=max_len):
  model.eval()
  cleaned=clean_text(text)
  seq=text_to_sequence(cleaned,vocab)
  padded=pad_sequence(seq,max_len)
  tensor_input=torch.tensor([padded],dtype=torch.long).to(device)

  with torch.no_grad():
    output=model(tensor_input)
    probs=torch.softmax(output,dim=1)
    confidence,predicted_class=torch.max(probs,dim=1)
  sentiment="Positive" if predicted_class.item()==1 else "Negative"
  return sentiment,confidence.item()


In [ ]:
sample_text = "The service was wonderful and amazing"
sentiment, score = predict_sentiment(sample_text, model, vocab)
print(f"Text: {sample_text}")
print(f"Prediction: {sentiment} (Confidence: {score:.2f})")

Text: The service was wonderful and amazing
Prediction: Positive (Confidence: 0.99)


In [ ]:
torch.save(model.state_dict(), 'bilstm_sentiment_model.pth')

In [ ]:
metadata = {
    'vocab': vocab,
    'max_len': max_len,
    'vocab_size': VOCAB_SIZE,
    'embedding_dim': EMBEDDING_DIM,
    'hidden_dim': HIDDEN_SIZE,
    'output_dim': OUTPUT_DIM
}

with open('vocab_metadata.pkl', 'wb') as f:
    pickle.dump(metadata, f)

print("Model and Metadata files saved successfully!")

Model and Metadata files saved successfully!
